# Test Suite: Feature Engineering & Multi-Modal FeatureUnion
This notebook executes the unit tests and interactive validations for `src/feature_engineering.py`.
It tests:
- `LinguisticFeatureTransformer` (9 stylometric features)
- Baseline Pipeline (Word-level TF-IDF)
- Enhanced Multi-Modal Pipeline (Word TF-IDF + Char TF-IDF + Stylometry + VADER Sentiment)


In [1]:
import os
import sys
import unittest
import numpy as np
import pandas as pd

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"Working Directory: {os.getcwd()}")
from src.feature_engineering import (
    LinguisticFeatureTransformer,
    build_baseline_feature_pipeline,
    build_enhanced_feature_pipeline
)
print("Feature Engineering module imported successfully!")


Working Directory: e:\ML\mini project 2
Feature Engineering module imported successfully!


## Interactive Demonstration 1: Stylometric Linguistic Feature Transformer


In [2]:
sample_texts = [
    "I LOVE this vacuum cleaner!! It did not lose suction once.",
    "Complete disaster. Stopped working in 10 minutes, do not buy.",
    "Standard paper notebook with 100 pages."
]

trans = LinguisticFeatureTransformer()
mat_ling = trans.fit_transform(sample_texts)
print(f"Linguistic matrix shape: {mat_ling.shape}")
print(f"Feature names: {trans.get_feature_names_out()}")
df_ling = pd.DataFrame(mat_ling, columns=trans.get_feature_names_out())
display(df_ling)


Linguistic matrix shape: (3, 9)
Feature names: ['ling_char_count', 'ling_word_count', 'ling_avg_word_length', 'ling_exclamation_count', 'ling_question_count', 'ling_uppercase_ratio', 'ling_punctuation_ratio', 'ling_first_person_pronouns', 'ling_lexical_diversity']


,ling_char_count,ling_word_count,ling_avg_word_length,ling_exclamation_count,ling_question_count,ling_uppercase_ratio,ling_punctuation_ratio,ling_first_person_pronouns,ling_lexical_diversity
0,58.0,11.0,4.091,2.0,0.0,0.1034,0.0517,1.0,1.0
1,61.0,10.0,4.900,0.0,0.0,0.0328,0.0492,0.0,1.0
2,39.0,6.0,5.500,0.0,0.0,0.0256,0.0256,0.0,1.0


## Interactive Demonstration 2: Baseline TF-IDF Pipeline


In [3]:
pipe_a = build_baseline_feature_pipeline(max_features=50)
mat_a = pipe_a.fit_transform(sample_texts)
print(f"Baseline TF-IDF matrix shape: {mat_a.shape}")
print(f"Sample vocabulary tokens: {list(pipe_a.vocabulary_.keys())[:10]}")


Baseline TF-IDF matrix shape: (3, 1)
Sample vocabulary tokens: ['not']


## Interactive Demonstration 3: Enhanced Multi-Modal FeatureUnion


In [4]:
pipe_b = build_enhanced_feature_pipeline(word_max_features=50, char_max_features=50)
mat_b = pipe_b.fit_transform(sample_texts)
print(f"Enhanced Multi-Modal matrix shape: {mat_b.shape}")
print(f"Total features extracted across all 4 modalities: {mat_b.shape[1]}")


Enhanced Multi-Modal matrix shape: (3, 47)
Total features extracted across all 4 modalities: 47


## Formal Unittest Suite Runner


In [5]:
class TestFeatureEngineering(unittest.TestCase):

    def setUp(self):
        self.sample_texts = [
            "I LOVE this vacuum cleaner!! It did not lose suction once.",
            "Complete disaster. Stopped working in 10 minutes, do not buy.",
            "Standard paper notebook with 100 pages."
        ]

    def test_linguistic_feature_transformer(self):
        trans = LinguisticFeatureTransformer()
        mat = trans.fit_transform(self.sample_texts)
        self.assertIsInstance(mat, np.ndarray)
        self.assertEqual(mat.shape, (3, 9))
        self.assertEqual(len(trans.get_feature_names_out()), 9)

    def test_baseline_feature_pipeline(self):
        pipe_a = build_baseline_feature_pipeline(max_features=50)
        mat_a = pipe_a.fit_transform(self.sample_texts)
        self.assertEqual(mat_a.shape[0], 3)
        self.assertGreater(mat_a.shape[1], 0)

    def test_enhanced_feature_pipeline(self):
        pipe_b = build_enhanced_feature_pipeline(
            word_max_features=50,
            char_max_features=50
        )
        mat_b = pipe_b.fit_transform(self.sample_texts)
        self.assertEqual(mat_b.shape[0], 3)
        self.assertGreater(mat_b.shape[1], 9 + 4)

# Run unittest suite in notebook
suite = unittest.TestLoader().loadTestsFromTestCase(TestFeatureEngineering)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Some tests failed!"


test_baseline_feature_pipeline (__main__.TestFeatureEngineering.test_baseline_feature_pipeline) ... ok
test_enhanced_feature_pipeline (__main__.TestFeatureEngineering.test_enhanced_feature_pipeline) ... ok
test_linguistic_feature_transformer (__main__.TestFeatureEngineering.test_linguistic_feature_transformer) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.013s

OK
